In [1]:
import os
import warnings
import importlib

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys
sys.path.append('/kaggle/input/datasets/keithmarange/rnn-methods/cmi_kaggle/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=RuntimeWarning)
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow logs
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using local data folder: C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\cmi_kaggle\data


In [3]:
model_run_name = 'rnn_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report   = False
save_model  = False
random_search = False

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.2,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'

raw_extractor = utils.RawSequenceExtractor(
    rotation_mode='delta_euler',   # angular velocity — most informative
)

preprocessor = ColumnTransformer([
    ('scale_signals', StandardScaler(), make_column_selector(pattern='acc|rot|thm')),
], remainder='drop', verbose_feature_names_out=False)
preprocessor.set_output(transform='pandas')

sequence_builder = utils.SequencePadder(maxlen=80, padding_value=0.0)
rnn_clf = utils.KerasRNNClassifier()

classifier = utils.ManyToOneWrapperRNN(
    estimator=rnn_clf,
    extractor=raw_extractor,
    target='gesture_action',
)

pipeline = Pipeline([
    ('raw_extractor', raw_extractor),
    ('preprocessor', preprocessor),
    ('sequence_builder', sequence_builder),
    ('classifier', classifier),
])

cv = GroupKFold(n_splits=3)

param_grid = {
    # RawSequenceExtractor params
    'raw_extractor__rotation_mode': ['euler'],
    'raw_extractor__acc_cols': [acc_columns],
    'raw_extractor__rot_cols': [rot_columns],
    'raw_extractor__thm_cols': [thm_columns],

    # SequencePadder params
    'sequence_builder__maxlen': [40],
    'sequence_builder__padding_value': [0.0],

    # RNN params (nested under classifier__estimator__)
    'classifier__estimator__rnn_type': ['rnn'],
    'classifier__estimator__rnn_units': [(64,)],
    'classifier__estimator__dense_units': [(16,)],
    'classifier__estimator__dropout': [0.1],
    'classifier__estimator__learning_rate': [1e-3],
    'classifier__estimator__batch_size': [16],
    'classifier__estimator__epochs': [30],
    'classifier__estimator__patience': [5],
}

In [6]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=5,  # LSTM is slow, keep iterations low
                cv=cv,
                random_state=42,
                n_jobs=-1,              # safer with tensorflow/keras on Kaggle
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=-1,              # safer with tensorflow/keras on Kaggle
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']
        
        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)
    
    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

Fitting 3 folds for each of 1 candidates, totalling 3 fits



In [7]:
if do_report:

    best_model = search_obj.best_estimator_

    extractor    = best_model.named_steps['imu_extractor']
    preprocessor = best_model.named_steps['preprocessor']
    classifier   = best_model.named_steps['classifier']

    X_feat = extractor.transform(test_sample_df)
    X_proc = preprocessor.transform(X_feat)

    y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
    y_true = y_true.reindex(X_proc.index)

    y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

    print(classification_report(y_true, y_pred))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).T.sort_values('f1-score', ascending=False)

    report_df.to_csv(model_run_folder_name + 'lstm_v1_per_gesture_scores.csv')